In [0]:
%sql
select * from zara_lakehouse.retail_bronze.discount_bronze


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df = spark.sql("select * from zara_lakehouse.retail_bronze.discount_bronze")

In [0]:
duplicates_df = df.groupBy("product_id", "city") \
    .count() \
    .filter("count > 1")

In [0]:
display(duplicates_df)

In [0]:
df_filtered = df \
             .filter(F.col("Product_id") == 113143)  \
             .filter(F.col("city") == "San Antonio") \
             .select("product_id", "city", "discount")
    
display(df_filtered)


In [0]:
window_spec = Window.partitionBy("product_id", "city").orderBy("discount")

df_result = df.withColumn("row_num", F.row_number().over(window_spec)) \
              .withColumn("rank_num", F.rank().over(window_spec)) \
              .withColumn("is_duplicate", F.col("row_num") > 1)

display(df_result)


In [0]:
df_result_filtered = df_result \
             .filter(F.col("Product_id") == 113143)  \
             .filter(F.col("city") == "San Antonio") \
             .select("product_id", "city", "discount", "row_num", "rank_num", "is_duplicate")
    
display(df_result_filtered)

In [0]:
total_rows = df.count()
print(total_rows)

In [0]:
df_result_count = df_result.groupBy("is_duplicate").count().show()

In [0]:
display (summary_df)

In [0]:
df_null = df_result.withColumn("is_id_null", F.col("product_id").isNull())

In [0]:
df_null_count = df_null.groupBy("is_id_null").count().show()

In [0]:
df_max = df.select(F.max("discount")).show()

In [0]:
df_min = df.select(F.min("discount")).show()

In [0]:
df_null = df_null \
            .withColumn("is_out_of_range", F.when((F.col("discount") < 0) | (F.col("discount") > 0.2), 1).otherwise(0))



In [0]:
display(df_out_of_range)

In [0]:
summary_df = df_null.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("is_duplicate") == False, 1).otherwise(0)).alias("non_duplicate_rows"),
    F.sum(F.when(F.col("is_duplicate") == True, 1).otherwise(0)).alias("duplicate_rows"),
    F.sum(F.when(F.col("is_id_null") == True, 1).otherwise(0)).alias("null_rows"),
    F.sum(F.when(F.col("is_id_null") == False, 1).otherwise(0)).alias("non_null_rows"),
    F.sum("is_out_of_range").alias("out_of_range_rows")
).withColumn(
    "ingestion_time", F.current_timestamp()
)

display (summary_df)

In [0]:
%sql
select count(*) from zara_lakehouse.retail_bronze.discount_bronze